# Chapter `1.4` - Multi-modal Agent

## Setup

### Module imports

In [1]:
# Basic utils.
import time
import base64

from os import getenv
from dotenv import load_dotenv

# o/p formatting
from pprint import pprint
from IPython.display import display, Markdown

# i/o modules
import io
from ipywidgets import FileUpload

from tqdm import tqdm
import sounddevice as sd
from scipy.io.wavfile import write

# LC modules
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain.messages import HumanMessage

### **Gemini** API setup

In [2]:
load_dotenv()

GOOGLE_API_KEY = getenv("GOOGLE_API_KEY")
GEMINI_API_MODEL = getenv("GEMINI_API_MODEL")

model = ChatGoogleGenerativeAI(model=GEMINI_API_MODEL, api_key=GOOGLE_API_KEY)
agent = create_agent(
    model=model,
    system_prompt="You are a multi-modal agent that can handle different modalities of input data from the user."
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


## Text input

In [3]:
question = HumanMessage(content=[
    {"type": "text", "text": "What is the capital of The Moon?"}
])

response = agent.invoke(
    {"messages": [question]}
)

Markdown(response['messages'][-1].content)

That's a fun question! The Moon doesn't have a capital city in the way that countries on Earth do. It's a natural celestial body, not a political entity with governments and capitals.

However, if we were to imagine a "capital" for the Moon, it would likely be a place of great significance, perhaps where humans first landed or where a future lunar base might be established. The **Sea of Tranquility** is a strong contender for this title, as it's where Apollo 11 landed and Neil Armstrong took his first steps.

So, while there's no official capital, the **Sea of Tranquility** holds a special, historical place in our exploration of the Moon!

## Image input

### Uploading an image file
![Trie Node Code Snippet](../../pics/trieNode.png)

> This is a `.png` format image.

In [4]:
uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [7]:
pprint(uploader.value)

({'content': <memory at 0x0000018C8BF4DC00>,
  'last_modified': datetime.datetime(2025, 4, 19, 16, 1, 59, 916000, tzinfo=datetime.timezone.utc),
  'name': 'trieNode.png',
  'size': 65213,
  'type': 'image/png'},)


### Encoding the image to `base64`
> This is done to aid comprehension for the **LLM**.

In [8]:
# Fetch the FIRST uploaded file
uploaded_file = uploader.value[0]

# This is a memoryview object
content_mv = uploaded_file["content"]

# Convert memoryview object -> bytes object
img_bytes = content_mv.tobytes()

# Now, perform base64 encoding
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

### Prompting

In [9]:
multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "What do you make of this image?"},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

Markdown(response['messages'][-1].content)

The image displays a code snippet that defines a `Node` class. Here's a breakdown of what it represents:

**Code:**

```java
class Node {
    Node one;
    Node zero;

    public Node() {
        one = null;
        zero = null;
    }
}
```

**Explanation:**

*   **`class Node { ... }`**: This declares a class named `Node`. In object-oriented programming, a class serves as a blueprint for creating objects.
*   **`Node one;`** and **`Node zero;`**: These lines declare two member variables (also called fields or attributes) of the `Node` class. Each of these variables is of type `Node` itself. This is a common pattern in data structures like linked lists or trees, where each node can point to other nodes.
*   **`public Node() { ... }`**: This is the constructor of the `Node` class. A constructor is a special method that is automatically called when a new `Node` object is created.
*   **`one = null;`** and **`zero = null;`**: Inside the constructor, these lines initialize the `one` and `zero` member variables to `null`. `null` represents the absence of a value or a reference to an object. This means that when a new `Node` is created, its `one` and `zero` pointers initially don't point to anything.

**In essence, this code defines a basic building block for creating linked data structures. Each `Node` object can potentially hold references to two other `Node` objects, and by default, these references are empty.**

The visual presentation with the colored dots (red, yellow, green) at the top left is typical of window controls in some operating systems or applications, suggesting that this code is displayed within a code editor or a similar environment. The dark background and syntax highlighting (colors for keywords like `class`, `public`, and `null`) are also common features of code editors to improve readability.

## Audio input

### Recording configuration

In [ ]:
duration = 5  # in seconds
sample_rate = 44100

print("🎤 Now, recording...")
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1)

# Progress bar for the duration
for _ in tqdm(range(duration * 15)):  # update 10x per second
    time.sleep(0.1)
sd.wait()

print("☑️ Recording finished.")

### Encoding the audio file

In [ ]:
# Write WAV to an in-memory buffer
buf = io.BytesIO()
write(buf, sample_rate, audio)
wav_bytes = buf.getvalue()

# Encode the audio file
aud_b64 = base64.b64encode(wav_bytes).decode("utf-8")

### Prompting

In [ ]:
multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "What you can make of this audio file? Gimme a report."},
    {"type": "audio", "base64": aud_b64, "mime_type": "audio/wav"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

Markdown(response['messages'][-1].content)